# ETL 项目管理台

这个 notebook 用来梳理当前仓库的启动方式，并集中执行常见管理动作：查看配置、列出作业、启动/停止 compose 环境、查看日志、手动触发 ETL。

## 当前项目如何启动

1. **容器编排**：`docker-compose.yml` 定义了 `postgres`、`redis`、`airflow-init`、`airflow-webserver`、`airflow-scheduler`、`airflow-worker`、`mysql-source`、`dolphindb`。
2. **手动执行**：`etl/__main__.py` 调用 `etl.cli:main`，可以直接执行 `python -m etl -c config/config.yaml [-j JOB]`。
3. **定时调度**：`dags/mysql_to_dolphindb.py` 在 Airflow 中按 `@hourly` 调度配置里的每个 job。
4. **镜像构建**：`Dockerfile` 基于 `apache/airflow:2.9.0-python3.10` 安装依赖并复制 `config/`、`etl/`、`dags/`、`scripts/`。

## 关键注意点

- `config/config.yaml` 当前默认连接的是外部 MySQL 与 DolphinDB 主机，而不是 compose 里启动的本地容器。
- 也就是说，**当前 compose 环境更像 Airflow 运行壳子**；ETL 真正连到哪里，取决于 YAML 配置。
- 如果要完全在本地 compose 环境联调，需要额外准备一份指向 `mysql-source` / `dolphindb` 的配置文件，或者把配置改成环境变量化。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import yaml
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docker-compose.yml").exists() and (candidate / "etl").exists():
            return candidate
    raise FileNotFoundError("未找到仓库根目录（缺少 docker-compose.yml 或 etl 目录）")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "config" / "config.yaml"
COMPOSE_FILE = REPO_ROOT / "docker-compose.yml"


def load_config():
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


CONFIG = load_config()
print(f"repo root: {REPO_ROOT}")
print(f"config path: {CONFIG_PATH}")


In [ ]:
def run(cmd, cwd: Path = REPO_ROOT, check: bool = False):
    print("$", " ".join(map(str, cmd)))
    completed = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(f"命令失败，退出码={completed.returncode}")
    return completed


def detect_compose_cmd():
    for base in (["docker", "compose"], ["docker-compose"]):
        probe = subprocess.run(base + ["version"], text=True, capture_output=True)
        if probe.returncode == 0:
            return base
    raise RuntimeError("未检测到 docker compose / docker-compose")


def compose(*args, check: bool = False):
    return run([*detect_compose_cmd(), "-f", COMPOSE_FILE, *args], check=check)


def etl(*args, check: bool = False):
    return run([sys.executable, "-m", "etl", "-c", CONFIG_PATH, *args], check=check)


In [ ]:
def startup_summary():
    jobs = CONFIG.get("jobs", [])
    summary = {
        "airflow_executor": "CeleryExecutor",
        "airflow_ui": "http://localhost:8080",
        "airflow_default_user": "admin",
        "airflow_default_password": "admin",
        "schedule": "@hourly",
        "job_count": len(jobs),
        "mysql_target": f"{CONFIG['mysql']['host']}:{CONFIG['mysql']['port']}",
        "dolphindb_target": f"{CONFIG['dolphindb']['host']}:{CONFIG['dolphindb']['port']}",
        "checkpoint_db": CONFIG["checkpoint"]["db_path"],
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


startup_summary()


In [ ]:
def job_rows():
    rows = []
    for job in CONFIG.get("jobs", []):
        rows.append({
            "name": job["name"],
            "mysql_table": job["mysql_table"],
            "dolphindb_table": job["dolphindb_table"],
            "incremental": job.get("incremental", False),
            "incremental_col": job.get("incremental_col"),
            "writer_strategy": job.get("writer_strategy"),
            "chunk_size": job.get("chunk_size"),
            "where": job.get("where", ""),
        })
    return rows


rows = job_rows()
if pd is not None:
    display(pd.DataFrame(rows))
else:
    for row in rows:
        print(row)


In [ ]:
LOCAL_MYSQL_HOSTS = {"mysql-source", "127.0.0.1", "localhost"}
LOCAL_DDB_HOSTS = {"dolphindb", "127.0.0.1", "localhost"}


def show_connection_warning():
    mysql_host = str(CONFIG["mysql"]["host"])
    ddb_host = str(CONFIG["dolphindb"]["host"])
    if mysql_host not in LOCAL_MYSQL_HOSTS or ddb_host not in LOCAL_DDB_HOSTS:
        print("注意：当前 YAML 配置默认连接的是外部服务，而不是 compose 内部服务。")
        print(f"MySQL 目标: {mysql_host}:{CONFIG['mysql']['port']}")
        print(f"DolphinDB 目标: {ddb_host}:{CONFIG['dolphindb']['port']}")
        print("如果要做本地联调，请另备一份指向 mysql-source / dolphindb 的配置文件。")
    else:
        print("当前配置已经指向本地 compose 服务。")


show_connection_warning()


## 常用管理动作

下面这些函数不会自动执行重操作，你可以按需调用。建议顺序：

1. 首次启动先执行 `init_airflow()`
2. 然后执行 `start_stack()`
3. 用 `ps()` / `logs(service)` 检查状态
4. 用 `list_jobs()` / `run_job(name)` 手动验证 ETL
5. 收尾时执行 `stop_stack()`


In [ ]:
def init_airflow():
    return compose("run", "--rm", "airflow-init", check=True)


def start_stack(detached: bool = True):
    args = ["up"]
    if detached:
        args.append("-d")
    args.extend([
        "postgres",
        "redis",
        "mysql-source",
        "dolphindb",
        "airflow-webserver",
        "airflow-scheduler",
        "airflow-worker",
    ])
    return compose(*args, check=True)


def stop_stack(remove_volumes: bool = False):
    args = ["down"]
    if remove_volumes:
        args.append("-v")
    return compose(*args, check=True)


def ps():
    return compose("ps", check=True)


def logs(service: str | None = None, tail: int = 100):
    args = ["logs", "--tail", str(tail)]
    if service:
        args.append(service)
    return compose(*args, check=True)


In [ ]:
def list_jobs():
    return etl("--list", check=True)


def run_job(job_name: str):
    return etl("-j", job_name, check=True)


def run_all_jobs():
    return etl(check=True)


In [ ]:
print("示例调用：")
print("  init_airflow()")
print("  start_stack()")
print("  ps()")
print("  logs('airflow-scheduler')")
print("  list_jobs()")
print("  run_job('secumain_sync')")
print("  stop_stack()")


## 补充说明

- Airflow UI 端口在 compose 中映射为 `8080:8080`，默认用户密码是 `admin / admin`。
- DAG ID 是 `mysql_to_dolphindb_etl`，每个 task 由 `config.yaml` 中的 `jobs` 动态生成。
- 如果 notebook 所在内核没有安装仓库依赖，可以先在对应环境安装 `requirements.txt`。
